# Project: What's the best item to dropship today?

Supplimental Question(s): 
- Can we know what might be the best thing to dropship a month from now?
- What are our data sources for purchase information/trends? How about wholesale distributor pricing and price change trends?

Givens: 
- While we can't anticipate drastic availability/price changes caused by political action, our data/model will focus on recurrent trends within the market in a similar state to how it is now. 
- Trends are equally as spontaneous (i.e Stanley Cups, Air Friers, Pop-Mart collectables) so prediction is considered impossible because the nature of them are based on intent (marketing, invested exposure, etc) and less so on existant market trends


Notes: 
- Utilize webscrapping on Amazon best sellers to analyze comnsumer purchase likelihood 
    - Pull image data along with feature data from best sellers 
- Use aliexpress and Temu to image search for corresponding dropship options 
    - Pull image data from top 3 searches
    - Use premade image comparison model to determine the best match 
    - take features of best fit image, include price, shipping 
- Create data report for categorical purchase points


## Scapeweb for Consumer Purchase Data

Categories are wide set so we'll use provided sales data to better lock-on our scraping!

In [1]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
import time
import random
import pyautogui
import undetected_chromedriver as uc
import urllib.robotparser
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from multiprocessing import Process, Queue, Manager
import pickle
from tqdm import tqdm
import threading
from datetime import datetime


KeyboardInterrupt



In [ ]:
def getHeader(requests=True):
    """
    We may benefit from changing our user agent in case certain sites may have protocol against multiple requests
    
    Parameter(s): 
        - requests (boolean): defaulted to True; this parameter signifies the return of a REQUESTS formatted header.
            If False, function will simply return a random user agent. 
    
    Return(s): header formated to "requests.get()" 
    
    """
    #100 possible user-agents
    user_agents = [
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
    'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
    'Mozilla/5.0 (iPhone; CPU iPhone OS 15_4 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/15.0 Mobile/15E148 Safari/604.1',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36 Edg/114.0.0.0',
    'Mozilla/5.0 (Linux; Android 11; SM-G981B) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Mobile Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36 OPR/90.0.0.0',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 12_6) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/110.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Linux; Android 12; Pixel 6) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Mobile Safari/537.36',
    'Mozilla/5.0 (iPad; CPU OS 16_2 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/16.0 Mobile/15E148 Safari/604.1',
    'Mozilla/5.0 (Windows NT 6.1; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/88.0.4324.182 Safari/537.36',
    'Mozilla/5.0 (X11; Ubuntu; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/89.0.4389.114 Safari/537.36',
    'Mozilla/5.0 (iPhone; CPU iPhone OS 14_8 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/14.0 Mobile/15E148 Safari/604.1',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:109.0) Gecko/20100101 Firefox/109.0',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/92.0.4515.131 Safari/537.36',
    'Mozilla/5.0 (Linux; Android 10; SM-N970U) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/92.0.4515.159 Mobile Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/95.0.4638.69 Safari/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_14_6) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/92.0.4515.159 Safari/537.36',
    'Mozilla/5.0 (Windows NT 6.1; WOW64; Trident/7.0; AS; rv:11.0) like Gecko',
    'Mozilla/5.0 (Linux; Android 9; SM-J415F) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.101 Mobile Safari/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_13_6) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/87.0.4280.88 Safari/537.36',
    'Mozilla/5.0 (Windows NT 6.3; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/89.0.4389.82 Safari/537.36',
    'Mozilla/5.0 (Linux; Android 7.0; SM-J727T1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/88.0.4324.181 Mobile Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/94.0.4606.81 Safari/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 11_2_3) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/90.0.4430.93 Safari/537.36',
    'Mozilla/5.0 (Linux; U; Android 9; en-US; moto e(6) play) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/88.0.4324.181 Mobile Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/89.0.4389.128 Safari/537.36',
    'Mozilla/5.0 (Linux; Android 11; SM-G988U) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/93.0.4577.62 Mobile Safari/537.36',
    'Mozilla/5.0 (iPhone; CPU iPhone OS 12_5_5 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/12.0 Mobile/15E148 Safari/604.1',
    'Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/85.0.4183.121 Safari/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_12_6) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/83.0.4103.116 Safari/537.36',
    'Mozilla/5.0 (Linux; Android 11; SM-A715F) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/92.0.4515.159 Mobile Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/101.0.4951.67 Safari/537.36',
    'Mozilla/5.0 (X11; Linux x86_64; rv:102.0) Gecko/20100101 Firefox/102.0',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_12_6) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/84.0.4147.135 Safari/537.36',
    'Mozilla/5.0 (Linux; Android 8.1.0; SM-J810Y) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/92.0.4515.131 Mobile Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; WOW64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/86.0.4240.198 Safari/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_6) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/89.0.4389.82 Safari/537.36',
    'Mozilla/5.0 (Linux; Android 7.1.1; SM-J530G) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/90.0.4430.66 Mobile Safari/537.36',
    'Mozilla/5.0 (Windows NT 6.1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/76.0.3809.132 Safari/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 11_1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/87.0.4280.141 Safari/537.36',
    'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/80.0.3987.132 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/113.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Linux; Android 9; SM-T590) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/92.0.4515.105 Safari/537.36',
    'Mozilla/5.0 (Windows NT 6.3; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/90.0.4430.212 Safari/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_14_5) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/75.0.3770.142 Safari/537.36',
    'Mozilla/5.0 (Linux; U; Android 10; en-us; SM-A205U) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/87.0.4280.141 Mobile Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/93.0.4577.63 Safari/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_13_3) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/70.0.3538.77 Safari/537.36',
    'Mozilla/5.0 (Linux; Android 11; SM-G973F) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/97.0.4692.98 Mobile Safari/537.36',
    'Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/55.0.2883.87 Safari/537.36',
    'Mozilla/5.0 (iPhone; CPU iPhone OS 13_7 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/13.0 Mobile/15E148 Safari/604.1',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_10_5) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/78.0.3904.108 Safari/537.36',
    'Mozilla/5.0 (Windows NT 6.1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/49.0.2623.112 Safari/537.36',
    'Mozilla/5.0 (Linux; Android 6.0.1; SM-T350) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/86.0.4240.198 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/83.0.4103.97 Safari/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_11_6) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/67.0.3396.99 Safari/537.36',
    'Mozilla/5.0 (Linux; Android 10; SM-G950F) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.120 Mobile Safari/537.36',
    'Mozilla/5.0 (Windows NT 6.1; WOW64; Trident/7.0; rv:11.0) like Gecko',
    'Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:98.0) Gecko/20100101 Firefox/98.0',
    'Mozilla/5.0 (Linux; Android 12; SM-F926B) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/96.0.4664.110 Mobile Safari/537.36',
    'Mozilla/5.0 (Windows NT 6.1; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/77.0.3865.90 Safari/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/79.0.3945.130 Safari/537.36',
    'Mozilla/5.0 (Linux; Android 7.0; HUAWEI NXT-L29) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/75.0.3770.143 Mobile Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:100.0) Gecko/20100101 Firefox/100.0',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_11_3) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/48.0.2564.109 Safari/537.36',
    'Mozilla/5.0 (Linux; Android 9; SAMSUNG SM-J330FN) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/76.0.3809.132 Mobile Safari/537.36',
    'Mozilla/5.0 (Windows NT 6.1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/57.0.2987.133 Safari/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_14_3) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/72.0.3626.121 Safari/537.36',
    'Mozilla/5.0 (Linux; Android 10; LM-X210APM) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/86.0.4240.185 Mobile Safari/537.36'
    ]
    
    if not requests:
        return random.choice(user_agents)
    
    return ({'User-Agent': random.choice(user_agents), 'Accept-Language': 'en-US, en;q=0.5'})


def random_mouse_move():
    """
    Moves the mouse randomly within the browser window to avoid unnatural/notably-automated action.
    
    Parameter(s): None
    
    Return(s): None
    """
    screen_width, screen_height = pyautogui.size()  # Get screen size
    for _ in range(random.randint(5, 10)):  # Move randomly 5-10 times
        x = random.randint(200, screen_width - 200)  # Avoid edges
        y = random.randint(200, screen_height - 200)
        pyautogui.moveTo(x, y, duration=random.uniform(0.2, 2.0))  # Random movement duration
        time.sleep(random.uniform(0.5, 1.5))  # Random delay
        
    return



def run_selenium(URL):
    """
        Attempt the html retrieval process using the google chrome-based library, selenium. 
        This function will run before all worker functions.
    
    Parameter(s):
        - URL (string): URL of webpage for our html retrieval process
        
    Return(s):
        - soup (BeautifulSoup Object): related soup from webpage. Will be None if connection failed.
        - connecting (boolean): True if webpage connection and html retrieval went well. False if not.
    
    """
    
    print("Running Alt. Driver: Selenium")
    connecting = False
    

    NETWORK_CONDITIONS = {
        "offline": False,
        "latency": 250,  # Lower the delay to 250ms
        "downloadThroughput": 3 * 1024 * 1024 / 8,  # 3 Mbps
        "uploadThroughput": 1.5 * 1024 * 1024 / 8,  # 1.5 Mbps
    }
    
    options = webdriver.ChromeOptions()
    options.binary_location = "C:/Program Files/Google/Chrome/Application/chrome.exe" # Point to your custom Chrome path
    options.add_argument("start-maximized")  # Start with a maximized window
    options.add_argument("--no-sandbox")  # Bypass OS security model (for some systems)
    options.add_argument("--disable-dev-shm-usage")  # Prevent memory issues
    options.add_argument("disable-infobars"); #disabling infobars
    options.add_argument("--disable-extensions"); #disabling extensions
    options.add_argument("--disable-gpu"); #applicable to windows os only
    options.add_argument('--remote-debugging-port=9222')
    options.set_capability("goog:loggingPrefs", {"performance": "ALL"})  # Enable performance logs
    options.add_argument("--disable-blink-features=AutomationControlled") 
 
    # setting the driver path and requesting a page 
    driver = webdriver.Chrome(options=options)
    
    # Load the stealth script
    with open("C:/Users/arias/Downloads/puppeteer-extra-raw-files/headful-chrome-stealth.js", "r") as f:
        stealth_script = f.read()

    # Inject the script before any page loads
    driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {"source": stealth_script})

    
    driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
        "source": """
            Object.defineProperty(navigator, 'platform', {get: () => 'Win32'});
            Object.defineProperty(navigator, 'language', {get: () => 'en-US'});
        """
    })
 
    # changing the property of the navigator value for webdriver to undefined 
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})") 
 
    # Enable network throttling; reduced likelyhood of request throttle
    driver.execute_cdp_cmd("Network.enable", {})
    driver.execute_cdp_cmd("Network.emulateNetworkConditions", NETWORK_CONDITIONS)

    
    driver.execute_cdp_cmd(
        "Network.setUserAgentOverride", {"userAgent": getHeader(requests=False)}
    )
    
    driver.get(URL)
    
    random_mouse_move()
    
    # Extract network logs
    logs = driver.get_log("performance")
    for log in logs:
        #print(log["message"])
        if '"Network.responseReceived"' in log["message"]:
            connecting = True
            
    if not connecting:
        print("Selenium Could Not Establish a Connection")
        return None, False
    
    soup = BeautifulSoup(driver.page_source, "html.parser")
    
    denied_string = "Request was throttled"
    if denied_string in soup.text:
        print(f"{URL}: Scapping Error: {denied_string}")
        connecting = False 
    
    time.sleep(random.uniform(2.0, 5.0))
    driver.quit()
    
    return soup, connecting

def compliance_check(urls, robot_txt_url):
    """
    Use reader to identify and cross-reference amazon's bot policy for compliance.
    
    Parameter(s):
        urls (array[strings]): an array of urls to be asesssed
        robot_txt_url (string): a url of the websites /robot.txt 
        
    Return(s): None
    
    """
    
    rp = urllib.robotparser.RobotFileParser()
    rp.set_url(robot_txt_url)  
    rp.read()
    compliant = True
    for url in urls:
        if not rp.can_fetch("*", url):
            print(f"Woah there, we're not allowed there: {url}")
            compliant = False
    if compliant:
        print("Robot.txt compliant...!")
    else: 
        print("Links to be accessed are not compliant with this website's policy.")
    return 

In [ ]:
def main():
    
    base_URL = "https://www.amazon.com/"
    URL = "https://www.amazon.com/gp/bestsellers/?ref_=nav_cs_bestsellers"

    time.sleep(5)  # Delay to prevent rate-limiting
    soup, _ = run_selenium(URL)

    if not _:
        print(f"Testing Soup: {soup}")
        
    all_item_tuples = []
    
    #We need all sidebar category links to scrape
    #Note that attributes are identified when visually inspecting the elements of the webpage
    tree_items = soup.find_all('div', role='treeitem')
    page_links = [item.find('a')['href'] for item in tree_items if item.find('a')]
    page_links = page_links[2:] #remove amazon product category & refurbished items
    #compliance_check(page_links, "https://www.amazon.com/robots.txt")
    for page_link in page_links:
        URL = base_URL + page_link
        time.sleep(random.uniform(2.0, 5.0))  # Delay to prevent rate-limiting
        cat_soup, _ = run_selenium(URL)
        category_title = cat_soup.find("h1", class_="a-size-large a-spacing-medium a-text-bold").text.strip()
        category = category_title.replace("Best Sellers in ", "").strip()              
        item_links = cat_soup.find_all("a", class_='a-link-normal aok-block')
        #compliance_check(item_links, "https://www.amazon.com/robots.txt")
        for item_link in item_links:
            all_item_tuples.append((base_URL + item_link, category_title))
        
    with open("amazon_item_tuples", "wb") as f:
        pickle.dump(all_item_tuples, f)
        print(f"All {len(all_item_tuples)} Item Links Saved!")
        
if __name__ == "__main__":
    main()

In [ ]:
def scrape_worker(queue, results, lock):

    while not queue.empty():
        try:
            idx, url, category = queue.get_nowait()

            try:
                item_soup, _ = runSelenium(url)

                dataset["category"].append(category)
                      
                title = item_soup.find('span', id='productTitle').text.strip()
                dataset["product title"].append(title)
        
                      
                dollar = item_soup.find('span', class_='a-price-whole').text.strip()
                cents = item_soup.find('span', class_='a-price-fraction').text.strip()
                price = float(dollar+cents)
                dataset["price"].append(price)
        
        
                image = item_soup.find('div', id='imgTagWrapperId').find('img')
                image_src = image['src']
                dataset["image"].append(image_src)
                      
                volume = item_soup.find('span', id='social-proofing-faceout-title-tk_bought')
                volume = item_soup.find('span', class_="a-text-bold").text.strip()
                volume = bought_last_month.replace("bought", "").strip()
                dataset["bought last month"].append(volume)
                
                
                
                result = {
                    "url": url,
                    "data": soup,
                    "status": "success",
                    "error": None,
                    "timestamp": datetime.utcnow().isoformat(),
                    "product title": title,
                    "category": category,
                    "price": price,
                    "image": img,
                    "bought last month": volume
                }

            except Exception as e:
                result = {
                    "url": url,
                    "data": None,
                    "status": "failed",
                    "error": str(e),
                    "timestamp": datetime.utcnow().isoformat()
                    "product title": None,
                    "category": None,
                    "price": None,
                    "image": None,
                    "bought last month": None
                }
                }

            with lock:
                results[idx] = result

        except Exception as outer_e:
            print(f"[Worker] Error: {outer_e}")

    driver.quit()    
    


def checkpoint_saver(results, interval=SAVE_EVERY):
    while True:
        time.sleep(interval)
        try:
            with open(CHECKPOINT_FILE, "wb") as f:
                pickle.dump(dict(results), f)
            print(f"[Checkpoint] Saved at {datetime.utcnow().isoformat()}")
        except Exception as e:
            print(f"[Checkpoint Error] {e}")
            
def main():
    
    with open("amazon_item_tuples", "rb") as f:
        all_item_tuples = pickle.load(f)

    NUM_PAGES = len(all_item_tuples)
    NUM_WORKERS = 4
    SAVE_EVERY = 30  #secs
    manager = Manager()
    queue = Queue()
    results = manager.dict()
    lock = manager.Lock()
    CHECKPOINT_FILE = "Amazon_Sales_Raw_Dataset"


    # Start progress monitor in a thread
    progress_thread = threading.Thread(
    target=progress_monitor, args=(results, len(all_item_tuples))
    )
    progress_thread.daemon = True
    progress_thread.start()


    # Load from checkpoint if available
    try:
        with open(CHECKPOINT_FILE, "rb") as f:
            old_results = pickle.load(f)
            for k in old_results:
                results[k] = old_results[k]
            done_idxs = set(old_results.keys())
            print(f"[Resume] Loaded checkpoint with {len(done_idxs)} entries.")
    except:
        done_idxs = set()
        print("[Start Fresh] No checkpoint found.")

    # Fill queue with (idx, url) tuples
    for idx, (item_link, category) in enumerate(all_item_tuples):
        if idx not in done_idxs:
            queue.put((idx, item_link, category))

    # Start checkpoint saver
    saver = Process(target=checkpoint_saver, args=(results,))
    saver.daemon = True
    saver.start()

    # Start workers
    workers = []
    for _ in range(NUM_WORKERS):
        p = Process(target=scrape_worker, args=(queue, results, lock))
        workers.append(p)
        p.start()

    for p in workers:
        p.join()

    # Final save
    with open(CHECKPOINT_FILE, "wb") as f:
        pickle.dump(dict(results), f)
    print("[Final Save] All done.")

    dataset = {
        "product title": [],
        "category": [],
        "price": [],
        "image": [],
        "bought last month": []
    }

if __name__ == "__main__":
    main()